In [ ]:
!apt-get install tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -r /content/drive/MyDrive/sports_video_summarizer/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.7 MB/s eta 0:00:00


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/')

In [ ]:
import os
import gradio as gr
from sports_video_summarizer.train import SportsVideoSummarizer

BASE_DIR = "/content/drive/MyDrive/sports_video_summarizer"
OUTPUT_DIR = os.path.join(BASE_DIR, "output_app")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def summarize_video(video_path, query, use_pretrained=True):
    """Process video and generate summary"""
    try:
        summarizer = SportsVideoSummarizer(
            video_path=video_path,
            output_dir=OUTPUT_DIR
        )

        if use_pretrained:
          model_path = os.path.join(BASE_DIR, "/output_app/models/summarizer_model")
          summarizer.load_model(model_path)

        frames, timestamps, audio_path, frame_features, scenes = summarizer.preprocess_video()

        summary = summarizer.generate_summary(
            query=query,
            frames=frames,
            timestamps=timestamps,
            audio_path=audio_path,
            frame_features=frame_features,
            scenes=scenes
        )

        return summary['summary_video_path']

    except Exception as e:
        return f"Error: {str(e)}"

with gr.Blocks(title="Sports Video Summarizer") as app:

    gr.Markdown("""
    # Sports Video Summarizer
    ### Extract key moments from sports videos based on your query
    """)

    with gr.Row():
        with gr.Column():
            video_input = gr.Video(
                label="Upload Video",
                sources=["upload"],
            )
            query_input = gr.Textbox(
                label="Search Query",
                placeholder="e.g., 'show all goals', 'basketball dunks'",
                info="Describe what highlights you want to extract"
            )
            pretrained_checkbox = gr.Checkbox(
                label="Use Pre-trained Models",
                value=True,
                info="Uncheck to use fine-tuned models if available"
            )
            submit_btn = gr.Button("Generate Highlights", variant="primary")

        with gr.Column():
            output_video = gr.Video(
                label="Generated Highlights",
                autoplay=True,
                interactive=False
            )
            status = gr.Textbox(
                label="Processing Status",
                interactive=False
            )

    def process_video(video, query, use_pretrained, progress=gr.Progress()):
        progress(0.1, desc="Starting processing...")
        try:
            progress(0.3, desc="Analyzing video content...")
            output_path = summarize_video(video, query, use_pretrained)

            if output_path.startswith("Error:"):
                progress(1.0, desc="Completed with errors")
                return None, output_path

            progress(0.9, desc="Finalizing highlights...")
            return output_path, "Highlights generated successfully!"

        except Exception as e:
            progress(1.0, desc="Completed with errors")
            return None, f"Error: {str(e)}"

    submit_btn.click(
        fn=process_video,
        inputs=[video_input, query_input, pretrained_checkbox],
        outputs=[output_video, status],
        show_progress="full",
        api_name="summarize"
    )

app.launch(
    debug=True,
    share=True,
    show_error=True
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2a5f90b6e5a98b7e5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
